In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gc, json, pickle, random, sys, time
import importlib
from dataclasses import asdict, dataclass
from datetime import datetime
from itertools import product
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from torch_geometric.data import HeteroData

/home/biffon/.conda/envs/phage_host_env_gnn_model/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# USER CONFIGURATION: MODIFY ACCORDINGLY

TASK_NAME = "host_unseen"        # set as "host_unseen", "phage_unseen", "both_unseen"
ADAPTATION_MODE = "full"         # set as "full" or "decoder_only"

BASE_DIR = Path("datasets/138strains_46phages")
INTERACTION_FILE = BASE_DIR / "split_graphs_interactions" / "graph2.csv"
OUTPUT_DIR = Path("adaptation_results/138strains_46phages/routed_models")

CV_RESULTS_DIR = Path("PH_q_LOGOCV_results")
TRAINED_MODELS_DIR = Path("PH_trained_models") / TASK_NAME
MODEL_CLASS_DIR = Path("../a_models")

MODEL_NAMES = ["AF3", "ESM2", "ESMC", "LucaOne", "BacFormer"]
Q_LIST = [0.999, 0.995, 0.990, 0.950, 0.900, 0.850, 0.800, 0.750]

In [3]:
#USER INPUT FILES: register host embeddings, phage embeddings, and similarity matrices for each model
MODEL_FILES = {
    "ESM2": {
        "host_embedding": BASE_DIR / "embeddings/ESM2/Beamu2023_ESM2_host_embeddings.csv",
        "phage_embedding": BASE_DIR / "embeddings/ESM2/Beamu2023_ESM2_phage_embeddings.csv",
        "host_similarity": BASE_DIR / "cosine_similarity/ESM2/host_host_cosine_similarity_esm2_centered_0to1.csv",
        "phage_similarity": BASE_DIR / "cosine_similarity/ESM2/phage_phage_cosine_similarity_esm2_centered_0to1.csv",
    },
    "ESMC": {
        "host_embedding": BASE_DIR / "embeddings/ESMC/Beamu2023_ESMC_host_embeddings.csv",
        "phage_embedding": BASE_DIR / "embeddings/ESMC/Beamu2023_ESMC_phage_embeddings.csv",
        "host_similarity": BASE_DIR / "cosine_similarity/ESMC/host_host_cosine_similarity_esmc_centered_0to1.csv",
        "phage_similarity": BASE_DIR / "cosine_similarity/ESMC/phage_cosine_similarity_esmc_centered_0to1.csv",
    },
    "LucaOne": {
        "host_embedding": BASE_DIR / "embeddings/LucaOne/Beamu2023_LucaOne_host_embeddings.csv",
        "phage_embedding": BASE_DIR / "embeddings/LucaOne/Beamu2023_LucaOne_phage_embeddings.csv",
        "host_similarity": BASE_DIR / "cosine_similarity/LucaOne/host_host_cosine_similarity_lucaone_centered_0to1.csv",
        "phage_similarity": BASE_DIR / "cosine_similarity/LucaOne/phage_cosine_similarity_lucaone_centered_0to1.csv",
    },
    "AF3": {
        "host_embedding": BASE_DIR / "embeddings/AF3/Beamu2023_AF3_host_embeddings.csv",
        "phage_embedding": BASE_DIR / "embeddings/AF3/Beamu2023_AF3_phage_embeddings.csv",
        "host_similarity": BASE_DIR / "cosine_similarity/AF3/host_host_cosine_similarity_af3_esmIF1_embeds_centered_0to1.csv",
        "phage_similarity": BASE_DIR / "cosine_similarity/AF3/phage_phage_cosine_similarity_af3_esmIF1_embeds_centered_0to1.csv",
    },
    "BacFormer": {
        "host_embedding": BASE_DIR / "embeddings/BacFormer/Beamu2023_BacFormer_host_embeddings.csv",
        "phage_embedding": BASE_DIR / "embeddings/BacFormer/Beamu2023_BacFormer_phage_embeddings.csv",
        "host_similarity": BASE_DIR / "cosine_similarity/BacFormer/host_cosine_similarity_Bacformer_centered_0to1_PHL.csv",
        "phage_similarity": BASE_DIR / "cosine_similarity/BacFormer/phage_cosine_similarity_Bacformer_centered_0to1_PHL.csv",
    },
}

In [4]:
#loading one numeric embedding table with cleaned row identifiers
def load_embedding_table(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, index_col=0)
    df.index = df.index.astype(str).str.strip()
    df = df.apply(pd.to_numeric, errors="coerce")

    if df.index.duplicated().any():
        raise ValueError(f"Duplicate embedding IDs in {csv_path}.")

    if df.isna().any().any():
        raise ValueError(f"Missing or non-numeric values in {csv_path}.")

    return df

In [5]:
# Reading all host and phage embeddings before selecting routed models
def load_all_embeddings() -> Dict[str, Dict[str, pd.DataFrame]]:
    embeddings: Dict[str, Dict[str, pd.DataFrame]] = {}
    for model_name in MODEL_NAMES:
        files = MODEL_FILES[model_name]
        embeddings[model_name] = {
            "host": load_embedding_table(files["host_embedding"]),
            "phage": load_embedding_table(files["phage_embedding"]),
        }

        host_shape = embeddings[model_name]["host"].shape
        phage_shape = embeddings[model_name]["phage"].shape
        print(f"{model_name}: hosts={host_shape}; phages={phage_shape}")

    return embeddings

In [ ]:
#reading pretrained LOGOCV performance for one model-quantile combination
def read_pretrained_result(model_name: str, q: float) -> Optional[Dict[str, Any]]:
    pickle_path = (CV_RESULTS_DIR / f"{model_name}_results_raw/logocv_results_{model_name}_q_{q:.3f}.pickle")
    if not pickle_path.exists():
        return None

    with open(pickle_path, "rb") as handle:
        result = pickle.load(handle)

    hit_curve = np.asarray(result.get("mean_hit_curve", []), dtype=float)
    return {
        "model": model_name,
        "quantile": float(result.get("q", q)),
        "pooled_auc": float(result.get("pooled_auc", np.nan)),
        "pooled_ap": float(result.get("pooled_ap", np.nan)),
        "mean_hit10": float(hit_curve[9]) if hit_curve.size >= 10 else np.nan,
        "result_pickle": pickle_path,
    }

In [ ]:
#selecting pretrained model with the highest performance by AUPRC and using Hit@10 as the tie-breaker
def select_best_models_per_quantile() -> pd.DataFrame:
    cv_rows = []
    for model_name in MODEL_NAMES:
        for q in Q_LIST:
            result = read_pretrained_result(model_name, q)
            if result is not None:
                cv_rows.append(result)

    if not cv_rows:
        raise FileNotFoundError(f"No pretrained LOGOCV result files were found in: {CV_RESULTS_DIR}")

    cv_results_df = pd.DataFrame(cv_rows)
    best_model_per_q_df = (
        cv_results_df
        .dropna(subset=["pooled_ap"])
        .sort_values(["quantile", "pooled_ap", "mean_hit10"], ascending=[False, False, False])
        .drop_duplicates("quantile")
        .rename(
            columns={
                "model": "selected_model",
                "pooled_auc": "selected_cv_auc",
                "pooled_ap": "selected_cv_auprc",
                "mean_hit10": "selected_cv_hit10",
            }
        )
        .reset_index(drop=True)
    )
    return best_model_per_q_df

In [ ]:
# resolving the checkpoint, scaler, and metadata files for one routed model
def resolve_pretrained_files(model_name: str, q: float) -> Dict[str, Path]:
    root = TRAINED_MODELS_DIR / model_name / "models_per_q"
    if not root.exists():
        raise FileNotFoundError(f"Pretrained model directory does not exist:\n{root}")

    model_dir = root / f"{model_name}_q_{q:.3f}"
    if not model_dir.exists():
        available_dirs = sorted(
            path.name
            for path in root.iterdir()
            if path.is_dir()
        )

        raise FileNotFoundError(
            f"No pretrained directory found for "
            f"model={model_name}, q={q:.3f}.\n"
            f"Expected:\n{model_dir}\n\n"
            f"Available directories:\n"
            + "\n".join(available_dirs)
        )

    files = {
        "model_file": (model_dir / f"final_model_{model_name}_q_{q:.3f}.pt"),
        "scaler_file": (model_dir / f"final_scalers_{model_name}_q_{q:.3f}.npz"),
        "meta_file": (model_dir / f"final_meta_{model_name}_q_{q:.3f}.json"),
    }

    for file_type, file_path in files.items():
        if not file_path.exists():
            available_files = sorted(
                path.name
                for path in model_dir.iterdir()
                if path.is_file()
            )

            raise FileNotFoundError(
                f"Missing {file_type} for "
                f"model={model_name}, q={q:.3f}.\n"
                f"Expected:\n{file_path}\n\n"
                f"Files available in {model_dir}:\n"
                + "\n".join(available_files)
            )

    return files

In [ ]:
# attaching checkpoints, scalers, and metadata to each selected route
def build_routes(best_model_per_q_df: pd.DataFrame) -> Tuple[Dict[float, Dict[str, Any]], pd.DataFrame]:
    route_by_q: Dict[float, Dict[str, Any]] = {}
    for row in best_model_per_q_df.itertuples(index=False):
        q = float(row.quantile)
        route = {
            "model_name": row.selected_model,
            "pretrained_auc": float(row.selected_cv_auc),
            "pretrained_auprc": float(row.selected_cv_auprc),
            "pretrained_hit10": float(row.selected_cv_hit10),
            "result_pickle": Path(row.result_pickle),
        }

        route.update(resolve_pretrained_files(row.selected_model, q))
        route_by_q[q] = route

    routing_df = pd.DataFrame(
        [
            {"quantile": q, **route}
            for q, route in sorted(
                route_by_q.items(),
                reverse=True,
            )
        ]
    )
    return route_by_q, routing_df

#### Main fine-tuning configutaions

In [10]:
#storing all local-adaptation settings in one configuration object
@dataclass
class AdaptationConfig:
    task_name: str = "host_unseen"
    adaptation_mode: str = "full"

    base_dir: Path = BASE_DIR
    interaction_file: Path = INTERACTION_FILE
    output_dir: Path = OUTPUT_DIR
    model_class_dir: Path = MODEL_CLASS_DIR

    q_list: Sequence[float] = tuple(Q_LIST)

    seed: int = 1005
    sample_frac: float = 1.0
    run_id: int = 1
    sample_seed: int = 46

    finetune_epochs: int = 50
    final_adapt_epochs: int = 100
    finetune_lr: float = 1e-3
    finetune_weight_decay: float = 5e-3
    edge_loss_chunk: int = 0
    positive_weight_cap: float = 10.0

    k_phage_phage: int = 10
    k_host_host: int = 10
    max_k: int = 50

    receive_only_heldout_hosts: bool = True
    receive_only_heldout_phages: bool = True

    standard_model_module: str = "GNN_model_inductive"
    decoder_keywords: Tuple[str, ...] = ("clf", "decoder")

    device: Optional[str] = None

In [11]:
# Supported experiment settings
TASK_MODES = {"host_unseen", "phage_unseen", "both_unseen"}
ADAPTATION_MODES = {"full", "decoder_only"}

In [ ]:
# initializing reproducibility, runtime tracking, device, and output directories.
def initialize_experiment(config: AdaptationConfig, routing_table: pd.DataFrame) -> Tuple[float, str]:

    global CFG
    global device

    global OUT_DIR
    global ROUTING_DIR
    global PREDICTION_DIR
    global RESULT_DIR
    global SUMMARY_DIR
    global FINAL_MODEL_DIR

    CFG = config
    if CFG.task_name not in TASK_MODES:
        raise ValueError(
            f"Invalid task_name: {CFG.task_name}. "
            f"Supported values: {sorted(TASK_MODES)}"
        )

    if CFG.adaptation_mode not in ADAPTATION_MODES:
        raise ValueError(
            f"Invalid adaptation_mode: {CFG.adaptation_mode}. "
            f"Supported values: {sorted(ADAPTATION_MODES)}"
        )

    if not 0.0 < CFG.sample_frac <= 1.0:
        raise ValueError(
            f"sample_frac must be in (0, 1], "
            f"got {CFG.sample_frac}"
        )

    run_start_time = time.time()
    run_start_datetime = datetime.now().isoformat(timespec="seconds")

    random.seed(CFG.seed)
    np.random.seed(CFG.seed)
    torch.manual_seed(CFG.seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CFG.seed)

    device = torch.device(
        CFG.device
        or ("cuda" if torch.cuda.is_available() else "cpu")
    )

    # The launcher supplies: run_XX/sample_XXX
    OUT_DIR = Path(CFG.output_dir)

    ROUTING_DIR = OUT_DIR / "routing"
    PREDICTION_DIR = OUT_DIR / "predictions"
    RESULT_DIR = OUT_DIR / "results_raw"
    SUMMARY_DIR = OUT_DIR / "summaries"
    FINAL_MODEL_DIR = OUT_DIR / "final_routed_models"

    for directory in [OUT_DIR, ROUTING_DIR, PREDICTION_DIR, RESULT_DIR, SUMMARY_DIR, FINAL_MODEL_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

    routing_table.to_csv(ROUTING_DIR / "best_pretrained_model_per_quantile.csv", index=False)

    print("Task:", CFG.task_name)
    print("Adaptation mode:", CFG.adaptation_mode)
    print("Run:", CFG.run_id)
    print("Sample fraction:", CFG.sample_frac)
    print("Sampling seed:", CFG.sample_seed)
    print("Training seed:", CFG.seed)
    print("Device:", device)
    print("Output:", OUT_DIR)

    return run_start_time, run_start_datetime

In [ ]:
# loading the interaction matrix and sample supervision pairs
def load_and_sample_interactions(config: AdaptationConfig) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str], List[str]]:
    interaction_data = pd.read_csv(config.interaction_file, index_col=0)
    interaction_data.index = (interaction_data.index.astype(str).str.strip())
    interaction_data.columns = (interaction_data.columns.astype(str).str.strip())
    interaction_data = interaction_data.apply(pd.to_numeric, errors="coerce")

    hosts_local = interaction_data.index.tolist()
    phages_local = interaction_data.columns.tolist()
    print(
        f"Interaction matrix: "
        f"shape={interaction_data.shape}; "
        f"hosts={len(hosts_local)}; "
        f"phages={len(phages_local)}"
    )

    display(interaction_data.head())
    labels_full_local = (
        interaction_data
        .stack(dropna=True)
        .rename("label")
        .reset_index()
    )

    labels_full_local.columns = ["host_id", "phage_id", "label"]
    labels_full_local["label"] = (
        pd.to_numeric(labels_full_local["label"])
        .gt(0.5)
        .astype(np.int8)
    )

    sampled_parts = []
    for label_value, group in labels_full_local.groupby("label"):
        if config.sample_frac >= 1.0:
            sampled = group.copy()
        else:
            n_sample = max(
                1,
                min(
                    len(group),
                    round(
                        len(group)
                        * config.sample_frac
                    ),
                ),
            )

            sampled = group.sample(n=n_sample, random_state=config.sample_seed)
        sampled_parts.append(sampled)

    labels_df_local = (pd.concat(sampled_parts, ignore_index=True).sample(frac=1, random_state=config.sample_seed).reset_index(drop=True))

    print("Observed interactions:", len(labels_full_local))
    print("Adaptation interactions:", len(labels_df_local))
    print("Positive interactions:", int(labels_df_local["label"].sum()))
    print("Negative interactions:", int((labels_df_local["label"] == 0).sum()))

    return (interaction_data, labels_full_local, labels_df_local, hosts_local, phages_local)

In [ ]:
# similarity grouping helper functions
# computing the quantile thresholds and greedy similarity groups for selected node IDs
def load_similarity_groups(similarity_path: Path, target_ids: List[str], q_list: Sequence[float]) -> Tuple[Dict[float, List[List[str]]], Dict[float, float]]:

    sim_df = pd.read_csv(similarity_path, index_col=0)
    sim_df.index = sim_df.index.astype(str).str.strip()
    sim_df.columns = sim_df.columns.astype(str).str.strip()
    sim_df = sim_df.apply(pd.to_numeric, errors="coerce")

    raw_ids = sim_df.index.tolist()
    matrix = sim_df.to_numpy(np.float32)
    off_diagonal = matrix[~np.eye(matrix.shape[0], dtype=bool)]

    thresholds = {
        float(q): float(np.quantile(off_diagonal, float(q)))
        for q in q_list
    }

    groups_by_q = {}
    for q, threshold in thresholds.items():
        assignments = np.full(len(raw_ids), -1, dtype=np.int32)
        group_id = 0

        for index in range(len(raw_ids)):
            if assignments[index] == -1:
                members = (matrix[index] >= threshold) & (assignments == -1)
                assignments[members] = group_id
                group_id += 1

        group_by_id = dict(zip(raw_ids, assignments.tolist()))
        groups = {}

        for node_id in target_ids:
            groups.setdefault(group_by_id[node_id], []).append(node_id)

        groups_by_q[q] = sorted(groups.values(), key=len, reverse=True)

    return groups_by_q, thresholds

In [15]:
# loading one routed quantile context (selected model embeddings, pretrained scaler, metadata, and groups)
def load_route_context(q: float) -> Dict[str, Any]:
    route = ROUTE_BY_Q[float(q)]
    model_name = route["model_name"]

    host_df = EMBEDDINGS[model_name]["host"].reindex(hosts)
    phage_df = EMBEDDINGS[model_name]["phage"].reindex(phages)
    if host_df.isna().any().any() or phage_df.isna().any().any():
        raise ValueError(f"Missing local embeddings for routed model {model_name}.")

    with np.load(route["scaler_file"]) as scalers:
        ph_mean = scalers["ph_mean"].astype(np.float32)
        ph_scale = scalers["ph_scale"].astype(np.float32)
        ho_mean = scalers["ho_mean"].astype(np.float32)
        ho_scale = scalers["ho_scale"].astype(np.float32)

    ph_scale = np.where(ph_scale == 0, 1.0, ph_scale)
    ho_scale = np.where(ho_scale == 0, 1.0, ho_scale)

    X_phage = ((phage_df.to_numpy(np.float32) - ph_mean) / ph_scale).astype(np.float32)
    X_host = ((host_df.to_numpy(np.float32) - ho_mean) / ho_scale).astype(np.float32)

    model_files = MODEL_FILES[model_name]
    host_groups, host_thresholds = load_similarity_groups(model_files["host_similarity"], hosts, CFG.q_list)
    phage_groups, phage_thresholds = load_similarity_groups(model_files["phage_similarity"], phages, CFG.q_list)
    metadata = json.loads(route["meta_file"].read_text())

    return {
        **route,
        "X_host": X_host,
        "X_phage": X_phage,
        "host_groups": host_groups,
        "phage_groups": phage_groups,
        "host_thresholds": host_thresholds,
        "phage_thresholds": phage_thresholds,
        "model_kwargs": metadata["model_kwargs"],
    }

In [16]:
#graph construction 

# building directed within-type cosine k-nearest-neighbour edges
def same_type_knn_edges(X: np.ndarray, k: int, source_mask: np.ndarray, target_mask: np.ndarray) -> np.ndarray:
    source_indices = np.where(source_mask)[0]
    target_indices = np.where(target_mask)[0]

    if len(source_indices) == 0 or len(target_indices) == 0 or k <= 0:
        return np.empty((2, 0), dtype=np.int64)

    X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    similarity = X_norm @ X_norm.T
    edges = []

    # each target receives messages from its most similar permitted sources
    for target_index in target_indices:
        candidate_sources = source_indices[source_indices != target_index]

        if len(candidate_sources) == 0:
            continue

        n_neighbors = min(k, len(candidate_sources))
        top_sources = candidate_sources[
            np.argsort(-similarity[candidate_sources, target_index])[:n_neighbors]
        ]

        edges.extend(
            (int(source_index), int(target_index))
            for source_index in top_sources
        )

    if not edges:
        return np.empty((2, 0), dtype=np.int64)

    return np.asarray(edges, dtype=np.int64).T

In [17]:
# heterogeneous data and supervision edges
# Convert within-type message-passing graphs and phage-host supervision into PyTorch objects.
PHAGE_MP = ("phage", "similar_to", "phage")
HOST_MP = ("host", "similar_to", "host")

def build_heterodata(context: Dict[str, Any], phage_edges: np.ndarray, host_edges: np.ndarray) -> HeteroData:
    data = HeteroData()
    data["phage"].x = torch.from_numpy(context["X_phage"]).float().to(device)
    data["host"].x = torch.from_numpy(context["X_host"]).float().to(device)
    data[PHAGE_MP].edge_index = torch.from_numpy(phage_edges).long().to(device)
    data[HOST_MP].edge_index = torch.from_numpy(host_edges).long().to(device)
    return data

def supervision_edges(df: pd.DataFrame) -> Tuple[torch.Tensor, torch.Tensor]:
    phage_to_index = {value: index for index, value in enumerate(phages)}
    host_to_index = {value: index for index, value in enumerate(hosts)}

    edge_index = np.vstack([df["phage_id"].map(phage_to_index), df["host_id"].map(host_to_index)]).astype(np.int64)
    labels = df["label"].to_numpy(np.float32)

    return (
        torch.from_numpy(edge_index).long().to(device),
        torch.from_numpy(labels).float().to(device),
    )

In [ ]:
# loading pretrained GNN mdoels
# constructing the standard pretrained GNN model and load its weights.
def load_pretrained_model(context: Dict[str, Any], data: HeteroData):
    if str(CFG.model_class_dir) not in sys.path:
        sys.path.append(str(CFG.model_class_dir))

    Model = importlib.import_module(CFG.standard_model_module).Model
    model = Model(data, **context["model_kwargs"]).to(device)
    state = torch.load(context["model_file"], map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    model.load_state_dict(state, strict=True)

    return model

In [ ]:
# Configure trainable parameters for full or decoder-only fine-tuning.
def configure_trainable_parameters(model) -> Tuple[List, Dict[str, float]]:
    for parameter in model.parameters():
        parameter.requires_grad = False

    for name, parameter in model.named_parameters():
        lower_name = name.lower()

        if CFG.adaptation_mode == "full":
            parameter.requires_grad = True

        elif CFG.adaptation_mode == "decoder_only":
            parameter.requires_grad = any(
                keyword in lower_name
                for keyword in CFG.decoder_keywords
            )

    trainable = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if not trainable:
        raise RuntimeError(f"No trainable parameters selected for mode={CFG.adaptation_mode}.")

    n_trainable = sum(
        parameter.numel()
        for parameter in trainable
    )
    n_total = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    print(
        f"Adaptation mode: {CFG.adaptation_mode} | "
        f"trainable parameters: {n_trainable:,}/{n_total:,} "
        f"({100 * n_trainable / n_total:.2f}%)"
    )

    return trainable, {
        "n_trainable_params": int(n_trainable),
        "n_total_params": int(n_total),
        "trainable_percent": float(100 * n_trainable / n_total),
    }

In [ ]:
# Fine-tuning one pretrained model and scoring labeled evaluation pairs
def fine_tune_model(model, data: HeteroData, edge_index: torch.Tensor, labels: torch.Tensor, epochs: int) -> Tuple[Any, Dict[str, Any]]:

    parameters, train_info = configure_trainable_parameters(model)

    n_positive = float(labels.sum().item())
    n_negative = float(labels.numel() - n_positive)
    positive_weight = min(n_negative / max(1.0, n_positive), CFG.positive_weight_cap)

    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([positive_weight], device=device))
    optimizer = optim.AdamW(
        parameters,
        lr=CFG.finetune_lr,
        weight_decay=CFG.finetune_weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=CFG.finetune_lr * 0.05,
    )

    start_time = time.time()
    loss_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(data, edge_index).view(-1)
        loss = loss_fn(logits, labels.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(parameters, 1.0)
        optimizer.step()
        scheduler.step()

        loss_history.append(float(loss.item()))

        if epoch in {1, epochs} or epoch % 25 == 0:
            print(f"epoch={epoch:03d}; loss={loss.item():.4f}; lr={optimizer.param_groups[0]['lr']:.2e}")

    train_info.update({
        "epochs": int(epochs),
        "final_loss": float(loss_history[-1]),
        "loss_history": loss_history,
        "positive_weight": float(positive_weight),
        "train_seconds": float(time.time() - start_time),
    })
    return model, train_info

@torch.no_grad()
def predict_pairs(model, data: HeteroData, edge_index: torch.Tensor) -> np.ndarray:
    model.eval()
    embeddings = model.encode(data)
    logits = model.decode(embeddings, edge_index)
    return torch.sigmoid(logits).cpu().numpy().astype(np.float32)

In [21]:
# generating task specific folds (host-unseen, phage-unseen, or both-unseen adaptation folds)
def make_adaptation_folds(q: float, context: Dict[str, Any]) -> List[Tuple]:

    if CFG.task_name == "host_unseen":
        combinations = [
            (host_group, [])
            for host_group in context["host_groups"][q]
        ]
    elif CFG.task_name == "phage_unseen":
        combinations = [
            ([], phage_group)
            for phage_group in context["phage_groups"][q]
        ]
    else:
        combinations = product(
            context["host_groups"][q],
            context["phage_groups"][q],
        )

    folds = []
    for fold_index, (host_group, phage_group) in enumerate(combinations):
        held_hosts = set(host_group)
        held_phages = set(phage_group)

        if CFG.task_name == "host_unseen":
            test_mask = labels_df["host_id"].isin(held_hosts)
        elif CFG.task_name == "phage_unseen":
            test_mask = labels_df["phage_id"].isin(held_phages)
        else:
            test_mask = (
                labels_df["host_id"].isin(held_hosts) & labels_df["phage_id"].isin(held_phages)
            )

        adaptation_mask = (
            ~labels_df["host_id"].isin(held_hosts) & ~labels_df["phage_id"].isin(held_phages)
        )

        test_df = labels_df[test_mask].copy()
        adaptation_df = labels_df[adaptation_mask].copy()

        if len(test_df) and len(adaptation_df):
            folds.append(
                (
                    fold_index,
                    held_hosts,
                    held_phages,
                    adaptation_df,
                    test_df,
                )
            )

    return folds

In [22]:
# EValuating ranked predictions
# Computing pooled and query-level AUC, AUPRC, Hit@K, precision, and recall.
def evaluate_predictions(predictions: pd.DataFrame) -> Dict[str, Any]:
    pooled_labels = predictions["label"].to_numpy(np.int32)
    pooled_scores = predictions["adapted_score"].to_numpy(np.float32)

    pooled_auc = (
        float(roc_auc_score(pooled_labels, pooled_scores))
        if len(np.unique(pooled_labels)) > 1
        else float("nan")
    )

    pooled_ap = (
        float(average_precision_score(pooled_labels, pooled_scores))
        if pooled_labels.sum() > 0
        else float("nan")
    )

    query_column = (
        "phage_id"
        if CFG.task_name == "phage_unseen"
        else "host_id"
    )

    hit_at_k = {k: [] for k in range(1, CFG.max_k + 1)}
    prec_at_k = {k: [] for k in range(1, CFG.max_k + 1)}
    rec_at_k = {k: [] for k in range(1, CFG.max_k + 1)}
    auc_per_query, ap_per_query = [], []

    for _, group in predictions.groupby(query_column):
        group = group.sort_values("adapted_score", ascending=False)
        ranked_labels = group["label"].to_numpy(np.int32)

        n_positive = int(ranked_labels.sum())
        n_negative = int(len(ranked_labels) - n_positive)

        if n_positive == 0:
            continue

        query_scores = group["adapted_score"].to_numpy(np.float32)
        ap_per_query.append(float(average_precision_score(ranked_labels, query_scores)))

        if n_negative > 0:
            auc_per_query.append(float(roc_auc_score(ranked_labels, query_scores)))

        cumulative = np.cumsum(ranked_labels)

        for k in range(1, CFG.max_k + 1):
            available = min(k, len(ranked_labels))
            found = int(cumulative[available - 1])

            hit_at_k[k].append(float(found > 0))
            prec_at_k[k].append(float(found / available))
            rec_at_k[k].append(float(found / n_positive))

    mean_hit_curve = np.array([
        np.mean(hit_at_k[k]) if hit_at_k[k] else np.nan
        for k in range(1, CFG.max_k + 1)
    ], dtype=np.float32)

    mean_prec_curve = np.array([
        np.mean(prec_at_k[k]) if prec_at_k[k] else np.nan
        for k in range(1, CFG.max_k + 1)
    ], dtype=np.float32)

    mean_rec_curve = np.array([
        np.mean(rec_at_k[k]) if rec_at_k[k] else np.nan
        for k in range(1, CFG.max_k + 1)
    ], dtype=np.float32)

    return {
        "pooled_labels": pooled_labels,
        "pooled_scores": pooled_scores,
        "pooled_auc": pooled_auc,
        "pooled_ap": pooled_ap,
        "hit_curve": mean_hit_curve.copy(),
        "mean_hit_curve": mean_hit_curve,
        "mean_prec_curve": mean_prec_curve,
        "mean_rec_curve": mean_rec_curve,
        "hit_at_k": hit_at_k,
        "prec_at_k": prec_at_k,
        "rec_at_k": rec_at_k,
        "auc": auc_per_query,
        "ap": ap_per_query,
    }

In [23]:
# Run one adaptation fold using within-type phage-phage and host-host message passing.
def run_adaptation_fold(q: float, context: Dict[str, Any], fold: Tuple) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    fold_index, held_hosts, held_phages, adaptation_df, test_df = fold

    host_train_mask = np.array(
        [host not in held_hosts for host in hosts],
        dtype=bool,
    )
    phage_train_mask = np.array(
        [phage not in held_phages for phage in phages],
        dtype=bool,
    )

    all_host_mask = np.ones(len(hosts), dtype=bool)
    all_phage_mask = np.ones(len(phages), dtype=bool)

    # Training graphs contain training nodes only.
    train_phage_edges = same_type_knn_edges(
        X=context["X_phage"],
        k=CFG.k_phage_phage,
        source_mask=phage_train_mask,
        target_mask=phage_train_mask,
    )
    train_host_edges = same_type_knn_edges(
        X=context["X_host"],
        k=CFG.k_host_host,
        source_mask=host_train_mask,
        target_mask=host_train_mask,
    )

    # During evaluation, held-out nodes receive from training nodes only.
    eval_phage_source_mask = (
        phage_train_mask
        if CFG.receive_only_heldout_phages
        else all_phage_mask
    )
    eval_host_source_mask = (
        host_train_mask
        if CFG.receive_only_heldout_hosts
        else all_host_mask
    )

    evaluation_phage_edges = same_type_knn_edges(
        X=context["X_phage"],
        k=CFG.k_phage_phage,
        source_mask=eval_phage_source_mask,
        target_mask=all_phage_mask,
    )
    evaluation_host_edges = same_type_knn_edges(
        X=context["X_host"],
        k=CFG.k_host_host,
        source_mask=eval_host_source_mask,
        target_mask=all_host_mask,
    )

    graph_checks = [
        (train_phage_edges, "empty training phage-phage graph"),
        (train_host_edges, "empty training host-host graph"),
        (evaluation_phage_edges, "empty evaluation phage-phage graph"),
        (evaluation_host_edges, "empty evaluation host-host graph"),
    ]

    for edges, message in graph_checks:
        if edges.shape[1] == 0:
            raise RuntimeError(f"Fold {fold_index}, q={q:.3f}: {message}.")

    # Training nodes must be the only sources and targets in training graphs.
    if not phage_train_mask[train_phage_edges[0]].all():
        raise RuntimeError("DATA LEAKAGE: held-out phages appear as training graph sources.")
    if not phage_train_mask[train_phage_edges[1]].all():
        raise RuntimeError("DATA LEAKAGE: held-out phages appear as training graph targets.")
    if not host_train_mask[train_host_edges[0]].all():
        raise RuntimeError("DATA LEAKAGE: held-out hosts appear as training graph sources.")
    if not host_train_mask[train_host_edges[1]].all():
        raise RuntimeError("DATA LEAKAGE: held-out hosts appear as training graph targets.")

    # Held-out nodes must not send messages during evaluation.
    if CFG.receive_only_heldout_phages:
        if (~phage_train_mask[evaluation_phage_edges[0]]).any():
            raise RuntimeError("DATA LEAKAGE: held-out phages appear as evaluation graph sources.")

    if CFG.receive_only_heldout_hosts:
        if (~host_train_mask[evaluation_host_edges[0]]).any():
            raise RuntimeError("DATA LEAKAGE: held-out hosts appear as evaluation graph sources.")

    # Confirm that all held-out nodes receive at least one message.
    if held_phages:
        held_phage_indices = {
            index
            for index, phage in enumerate(phages)
            if phage in held_phages
        }
        evaluation_phage_targets = set(
            evaluation_phage_edges[1].astype(np.int64).tolist()
        )
        missing_phages = held_phage_indices - evaluation_phage_targets

        if missing_phages:
            raise RuntimeError(f"{len(missing_phages)} held-out phages receive no messages.")

    if held_hosts:
        held_host_indices = {
            index
            for index, host in enumerate(hosts)
            if host in held_hosts
        }
        evaluation_host_targets = set(
            evaluation_host_edges[1].astype(np.int64).tolist()
        )
        missing_hosts = held_host_indices - evaluation_host_targets

        if missing_hosts:
            raise RuntimeError(f"{len(missing_hosts)} held-out hosts receive no messages.")

    # Confirm that decoder supervision contains training nodes only.
    if adaptation_df["host_id"].isin(held_hosts).any():
        raise RuntimeError("DATA LEAKAGE: adaptation supervision contains held-out hosts.")

    if adaptation_df["phage_id"].isin(held_phages).any():
        raise RuntimeError("DATA LEAKAGE: adaptation supervision contains held-out phages.")

    train_data = build_heterodata(
        context=context,
        phage_edges=train_phage_edges,
        host_edges=train_host_edges,
    )
    evaluation_data = build_heterodata(
        context=context,
        phage_edges=evaluation_phage_edges,
        host_edges=evaluation_host_edges,
    )

    train_edge_index, train_labels = supervision_edges(adaptation_df)
    test_edge_index, _ = supervision_edges(test_df)

    model = load_pretrained_model(context, train_data)
    model, train_info = fine_tune_model(
        model=model,
        data=train_data,
        edge_index=train_edge_index,
        labels=train_labels,
        epochs=CFG.finetune_epochs,
    )

    predictions = test_df.copy()
    predictions["adapted_score"] = predict_pairs(
        model,
        evaluation_data,
        test_edge_index,
    )
    predictions["quantile"] = float(q)
    predictions["fold"] = int(fold_index)
    predictions["selected_model"] = context["model_name"]

    train_info.update({
        "n_train_phage_edges": int(train_phage_edges.shape[1]),
        "n_train_host_edges": int(train_host_edges.shape[1]),
        "n_evaluation_phage_edges": int(evaluation_phage_edges.shape[1]),
        "n_evaluation_host_edges": int(evaluation_host_edges.shape[1]),
        "n_adaptation_pairs": int(len(adaptation_df)),
        "n_test_pairs": int(len(test_df)),
    })

    del model, train_data, evaluation_data
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return predictions, train_info

In [24]:
# Run routed cross-validation for all configured quantiles.
def run_routed_cross_validation() -> Tuple[List[pd.DataFrame], Dict[float, Dict[str, Any]], Dict[float, List[Dict[str, Any]]], float]:

    cv_start_time = time.time()
    all_prediction_tables = []
    all_results: Dict[float, Dict[str, Any]] = {}
    fold_train_info: Dict[float, List[Dict[str, Any]]] = {}

    for q_value in CFG.q_list:
        q = float(q_value)
        context = load_route_context(q)
        folds = make_adaptation_folds(q, context)
        fold_train_info[q] = []
        quantile_predictions = []
        n_groups = len(folds)
        print(
            f"\n{'=' * 78}\n"
            f"Quantile: {q:.3f}; "
            f"Selected model: {context['model_name']} | "
            f"Similarity groups: {n_groups}; "
            f"Training groups: {max(0, n_groups - 1)}; "
            f"Held-out groups: 1 per fold "
            f"({n_groups} folds)\n"
            f"{'=' * 78}"
        )

        if not folds:
            raise RuntimeError(
                f"No valid adaptation folds were generated "
                f"for q={q:.3f}."
            )

        for fold in folds:
            predictions, training_info = (run_adaptation_fold(q, context, fold))
            quantile_predictions.append(predictions)
            all_prediction_tables.append(predictions)
            fold_train_info[q].append(
                {
                    "fold": int(fold[0]),
                    **training_info,
                }
            )

        quantile_df = pd.concat(quantile_predictions, ignore_index=True)
        metrics = evaluate_predictions(quantile_df)

        all_results[q] = {
            **metrics,
            "q": q,
            "selected_model": context["model_name"],
            "selected_cv_auc": context["pretrained_auc"],
            "selected_cv_auprc": context["pretrained_auprc"],
            "selected_cv_hit10": context["pretrained_hit10"],
            "adaptation_mode": CFG.adaptation_mode,
            "task_name": CFG.task_name,
            "fold_train_info": fold_train_info[q],
            "pair_rows": quantile_df.to_dict("records"),
        }
        print(f"[DONE] q={q:.3f}; model={context['model_name']}; AUC={metrics['pooled_auc']:.3f}; AUPRC={metrics['pooled_ap']:.3f}")
    cv_seconds = time.time() - cv_start_time
    return (all_prediction_tables, all_results, fold_train_info, cv_seconds)

In [25]:
# Save held-out predictions and per-quantile evaluation results.
def save_cross_validation_outputs(all_prediction_tables: List[pd.DataFrame], all_results: Dict[float, Dict[str, Any]],) -> Tuple[
    pd.DataFrame,
    pd.DataFrame,
    Path,
    Path,
]:
    if not all_prediction_tables:
        raise RuntimeError("No cross-validation predictions were generated.")

    all_predictions_df = pd.concat(all_prediction_tables, ignore_index=True)
    predictions_file = (PREDICTION_DIR / (f"{CFG.task_name}_{CFG.adaptation_mode}_heldout_predictions.csv"))
    all_predictions_df.to_csv(predictions_file, index=False)
    expected_predictions = (len(labels_df) * len(CFG.q_list))

    print(f"Observed held-out predictions: {len(all_predictions_df):,}")
    print(f"Expected held-out predictions: {expected_predictions:,}")

    if len(all_predictions_df) != expected_predictions:
        print("Warning: prediction count differs from n_interactions × n_quantiles.")

    pair_counts = (all_predictions_df.groupby(["quantile", "host_id", "phage_id"]).size())

    if not pair_counts.eq(1).all():
        duplicated_pairs = pair_counts[pair_counts.ne(1)]
        raise RuntimeError(f"Found {len(duplicated_pairs)} pair-quantile combinations that do not occur exactly once.")

    summary_rows = []
    for q_value in CFG.q_list:
        q = float(q_value)

        result = all_results[q]
        context = load_route_context(q)
        model_name = context["model_name"]

        result_file = (RESULT_DIR / (f"adapted_{model_name}_{CFG.task_name}_q_{q:.3f}.pickle"))
        with result_file.open("wb") as handle:
            pickle.dump(result, handle)

        quantile_predictions = (all_predictions_df.loc[np.isclose(all_predictions_df["quantile"], q)].copy())
        quantile_prediction_file = (PREDICTION_DIR / (f"adapted_predictions_{model_name}_q_{q:.3f}.csv"))
        quantile_predictions.to_csv(quantile_prediction_file, index=False,)
        hit10_values = result["hit_at_k"].get(10, [])

        summary_rows.append(
            {
                "quantile": q,
                "selected_model": model_name,
                "pretrained_auc": context["pretrained_auc"],
                "pretrained_auprc": context["pretrained_auprc"],
                "pretrained_hit10": context["pretrained_hit10"],
                "adapted_auc": result["pooled_auc"],
                "adapted_auprc": result["pooled_ap"],
                "adapted_hit10": (
                    float(np.mean(hit10_values))
                    if len(hit10_values) > 0
                    else np.nan
                ),
                "adaptation_mode": (CFG.adaptation_mode),
                "run_id": CFG.run_id,
                "sample_frac": CFG.sample_frac,
                "sample_seed": CFG.sample_seed,
                "training_seed": CFG.seed,
                "n_folds": len(result["fold_train_info"]),
                "n_predictions": len(quantile_predictions),
                "result_pickle": str(result_file),
                "prediction_file": str(quantile_prediction_file),
            }
        )

    summary_df = (pd.DataFrame(summary_rows).sort_values("quantile", ascending=False).reset_index(drop=True))
    summary_file = (SUMMARY_DIR / (f"{CFG.task_name}_{CFG.adaptation_mode}_pretrained_vs_adapted_summary.csv"))
    summary_df.to_csv(summary_file, index=False)
    display(summary_df)

    print("Combined predictions:", predictions_file)
    print("Performance summary:", summary_file)
    print("LOGOCV adaptation results saved successfully.")

    return (all_predictions_df, summary_df, predictions_file, summary_file)

In [ ]:
# Fine-tuning one final deployment model per quantile using all sampled local interactions.
def run_final_adaptation() -> Tuple[Dict[str, Dict[str, str]], Path, Path]:
    deployment_routes = {}
    all_final_losses = {}

    all_phage_mask = np.ones(len(phages), dtype=bool)
    all_host_mask = np.ones(len(hosts), dtype=bool)
    all_edge_index, all_labels = supervision_edges(labels_df)

    for q_value in CFG.q_list:
        q = float(q_value)
        context = load_route_context(q)
        model_name = context["model_name"]

        print(f"\n[FINAL ADAPTATION] q={q:.3f} | model={model_name} | mode={CFG.adaptation_mode}")

        final_phage_edges = same_type_knn_edges(
            X=context["X_phage"],
            k=CFG.k_phage_phage,
            source_mask=all_phage_mask,
            target_mask=all_phage_mask,
        )
        final_host_edges = same_type_knn_edges(
            X=context["X_host"],
            k=CFG.k_host_host,
            source_mask=all_host_mask,
            target_mask=all_host_mask,
        )

        if final_phage_edges.shape[1] == 0:
            raise RuntimeError(f"Final phage-phage graph is empty for q={q:.3f}.")

        if final_host_edges.shape[1] == 0:
            raise RuntimeError(f"Final host-host graph is empty for q={q:.3f}.")

        final_data = build_heterodata(
            context=context,
            phage_edges=final_phage_edges,
            host_edges=final_host_edges,
        )

        final_model = load_pretrained_model(context, final_data)
        final_model, final_train_info = fine_tune_model(
            model=final_model,
            data=final_data,
            edge_index=all_edge_index,
            labels=all_labels,
            epochs=CFG.final_adapt_epochs,
        )

        loss_history = final_train_info.get("loss_history")
        if loss_history is None:
            raise KeyError("fine_tune_model() must return 'loss_history'.")

        final_train_info.update({
            "message_passing_graph": "within_type_cosine_knn",
            "k_phage_phage": int(CFG.k_phage_phage),
            "k_host_host": int(CFG.k_host_host),
            "n_phage_phage_edges": int(final_phage_edges.shape[1]),
            "n_host_host_edges": int(final_host_edges.shape[1]),
        })

        all_final_losses[q] = list(loss_history)

        model_dir = FINAL_MODEL_DIR / f"{model_name}_q_{q:.3f}"
        model_dir.mkdir(parents=True, exist_ok=True)

        model_file = (model_dir / f"final_adapted_model_{model_name}_q_{q:.3f}.pt")
        scaler_file = (model_dir / f"final_scalers_{model_name}_q_{q:.3f}.npz")
        meta_file = (model_dir / f"final_meta_{model_name}_q_{q:.3f}.json")

        torch.save(final_model.state_dict(), model_file)

        with np.load(context["scaler_file"]) as scalers:
            np.savez(
                scaler_file,
                ph_mean=scalers["ph_mean"],
                ph_scale=scalers["ph_scale"],
                ho_mean=scalers["ho_mean"],
                ho_scale=scalers["ho_scale"],
            )

        final_metadata = {
            "model": model_name,
            "q": q,
            "task_name": CFG.task_name,
            "adaptation_mode": CFG.adaptation_mode,
            "run_id": CFG.run_id,
            "sample_frac": CFG.sample_frac,
            "sample_seed": CFG.sample_seed,
            "training_seed": CFG.seed,
            "message_passing_graph": "within_type_cosine_knn",
            "phage_relation": list(PHAGE_MP),
            "host_relation": list(HOST_MP),
            "k_phage_phage": int(CFG.k_phage_phage),
            "k_host_host": int(CFG.k_host_host),
            "model_kwargs": context["model_kwargs"],
            "configuration": asdict(CFG),
            "pretrained_model_file": str(context["model_file"]),
            "pretrained_scaler_file": str(context["scaler_file"]),
            "pretrained_meta_file": str(context["meta_file"]),
            "final_train_info": final_train_info,
            "n_hosts": len(hosts),
            "n_phages": len(phages),
            "n_phage_phage_edges": int(final_phage_edges.shape[1]),
            "n_host_host_edges": int(final_host_edges.shape[1]),
            "n_interactions": len(labels_df),
            "n_positive": int(labels_df["label"].sum()),
            "n_negative": int(
                len(labels_df) - labels_df["label"].sum()
            ),
        }

        meta_file.write_text(
            json.dumps(final_metadata, indent=2, default=str),
            encoding="utf-8",
        )

        deployment_routes[str(q)] = {
            "model_name": model_name,
            "model_file": str(model_file),
            "scaler_file": str(scaler_file),
            "meta_file": str(meta_file),
        }

        print("SAVED", model_file)

        del final_model, final_data
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    loss_df = pd.DataFrame({
        f"q_{q:.3f}": pd.Series(losses, index=np.arange(1, len(losses) + 1))
        for q, losses in all_final_losses.items()
    })

    loss_df.index.name = "epoch"
    loss_file = FINAL_MODEL_DIR / "final_adaptation_losses.csv"
    loss_df.to_csv(loss_file)

    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    for column in loss_df.columns:
        ax.plot(
            loss_df.index,
            loss_df[column],
            linewidth=1.5,
            label=column.replace("_", "="),
        )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Training loss")
    ax.set_title("Final adaptation loss per quantile")
    ax.legend(
        title="Quantile",
        frameon=False,
        ncol=2,
        fontsize=8,
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()

    loss_figure = FINAL_MODEL_DIR / "final_adaptation_losses.png"
    fig.savefig(loss_figure, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Loss file:", loss_file)
    print("Loss figure:", loss_figure)

    return deployment_routes, loss_file, loss_figure

In [ ]:
# saving  the routing manifest and runtime report.
def save_run_manifest(deployment_routes: Dict[str, Dict[str, str]],predictions_file: Path, summary_file: Path, loss_file: Path, 
                      loss_figure: Path, run_start_time: float, run_start_datetime: str, cv_seconds: float) -> Path:

    run_seconds = (time.time() - run_start_time)

    manifest = {
        "task_name": CFG.task_name,
        "adaptation_mode": CFG.adaptation_mode,
        "run_id": CFG.run_id,
        "sample_frac": CFG.sample_frac,
        "sample_seed": CFG.sample_seed,
        "training_seed": CFG.seed,
        "interaction_file": str(CFG.interaction_file),
        "selection_metric": ("pretrained pooled AUPRC"),
        "selection_tiebreaker": ("pretrained mean Hit@10"),
        "routes": deployment_routes,
        "predictions_file": str(predictions_file),
        "summary_file": str(summary_file),
        "final_loss_file": str(loss_file),
        "final_loss_figure": str(loss_figure),
        "run_start_datetime": (run_start_datetime),
        "run_end_datetime": (datetime.now().isoformat(timespec="seconds")),
        "cv_seconds": float(cv_seconds),
        "cv_minutes": float(cv_seconds / 60),
        "total_seconds": float(run_seconds),
        "total_minutes": float(run_seconds / 60),
    }

    manifest_file = (FINAL_MODEL_DIR / "routed_adaptation_manifest.json")
    manifest_file.write_text(
        json.dumps(manifest, indent=2, default=str),
        encoding="utf-8",
    )
    print(f"Cross-validation runtime: {cv_seconds / 60:.2f} minutes")
    print(f"Total runtime: {run_seconds / 60:.2f} minutes")

    return manifest_file


# public framework function (imported by the launcher)
def run_local_adaptation(config: AdaptationConfig,) -> Dict[str, Any]:

    global CFG
    global EMBEDDINGS
    global ROUTE_BY_Q
    global routing_df

    global interaction_data
    global labels_full
    global labels_df
    global hosts
    global phages

    CFG = config

    print("\n" + "=" * 80)
    print("STARTING ROUTED LOCAL ADAPTATION")
    print(f"Task            : {CFG.task_name}")
    print(f"Adaptation mode : {CFG.adaptation_mode}")
    print(f"Run             : {CFG.run_id}")
    print(f"Sample fraction : {CFG.sample_frac:.2f}")
    print(f"Training seed   : {CFG.seed}")
    print(f"Sampling seed   : {CFG.sample_seed}")
    print(f"Output          : {CFG.output_dir}")
    print("=" * 80)

    # Static model resources
    EMBEDDINGS = load_all_embeddings()
    best_model_per_q_df = (select_best_models_per_quantile())
    ROUTE_BY_Q, routing_df = build_routes(best_model_per_q_df)
    display(best_model_per_q_df)
    display(routing_df)

    # Initialize the current experiment
    (run_start_time, run_start_datetime) = initialize_experiment(CFG, routing_df)

    # Load and sample interactions
    (interaction_data, labels_full, labels_df, hosts, phages) = load_and_sample_interactions(CFG)

    # Routed LOGOCV
    (
        all_prediction_tables,
        all_results,
        fold_train_info,
        cv_seconds,
    ) = run_routed_cross_validation()

    (
        all_predictions_df,
        summary_df,
        predictions_file,
        summary_file,
    ) = save_cross_validation_outputs(
        all_prediction_tables,
        all_results,
    )

    # Final deployment models
    (
        deployment_routes,
        loss_file,
        loss_figure,
    ) = run_final_adaptation()

    manifest_file = save_run_manifest(
        deployment_routes=deployment_routes,
        predictions_file=predictions_file,
        summary_file=summary_file,
        loss_file=loss_file,
        loss_figure=loss_figure,
        run_start_time=run_start_time,
        run_start_datetime=run_start_datetime,
        cv_seconds=cv_seconds,
    )

    print("\nSaved outputs")
    print("Predictions:", predictions_file)
    print("Summary:", summary_file)
    print("Losses:", loss_file)
    print("Loss figure:", loss_figure)
    print("Manifest:", manifest_file)

    print("\nRouted local fine-tuning completed successfully.")

    return {
        "predictions_file": predictions_file,
        "summary_file": summary_file,
        "loss_file": loss_file,
        "loss_figure": loss_figure,
        "manifest_file": manifest_file,
        "all_predictions_df": (
            all_predictions_df
        ),
        "summary_df": summary_df,
        "all_results": all_results,
        "fold_train_info": fold_train_info,
        "deployment_routes": deployment_routes,
        "cv_seconds": cv_seconds,
    }

In [ ]:
# RUNNING FINE-TUNING
if __name__ == "__main__":
    notebook_config = AdaptationConfig(
        task_name=TASK_NAME,
        adaptation_mode=ADAPTATION_MODE,
        interaction_file=INTERACTION_FILE,
        output_dir=OUTPUT_DIR / TASK_NAME / ADAPTATION_MODE,
        model_class_dir=MODEL_CLASS_DIR,
    )

    notebook_results = run_local_adaptation(notebook_config)


STARTING ROUTED LOCAL ADAPTATION
Task            : host_unseen
Adaptation mode : full
Run             : 1
Sample fraction : 1.00
Training seed   : 1005
Sampling seed   : 46
Output          : adaptation_results/138strains_46phages/routed_models/host_unseen/full
AF3: hosts=(138, 512); phages=(46, 512)
ESM2: hosts=(138, 1280); phages=(46, 1280)
ESMC: hosts=(138, 1152); phages=(46, 1152)
LucaOne: hosts=(138, 2560); phages=(46, 2560)
BacFormer: hosts=(138, 480); phages=(46, 480)


,selected_model,quantile,selected_cv_auc,selected_cv_auprc,selected_cv_hit10,result_pickle
0,ESMC,0.999,0.907413,0.772472,0.859504,PH_q_LOGOCV_results/ESMC_results_raw/logocv_re...
1,BacFormer,0.995,0.904478,0.774684,0.859504,PH_q_LOGOCV_results/BacFormer_results_raw/logo...
2,BacFormer,0.990,0.903027,0.774667,0.851240,PH_q_LOGOCV_results/BacFormer_results_raw/logo...
3,ESMC,0.950,0.891284,0.721186,0.834711,PH_q_LOGOCV_results/ESMC_results_raw/logocv_re...
4,AF3,0.900,0.892769,0.740509,0.834711,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...
5,AF3,0.850,0.874372,0.707503,0.834711,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...
6,AF3,0.800,0.874844,0.740536,0.793388,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...
7,AF3,0.750,0.888613,0.729488,0.851240,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...


,quantile,model_name,pretrained_auc,pretrained_auprc,pretrained_hit10,result_pickle,model_file,scaler_file,meta_file
0,0.999,ESMC,0.907413,0.772472,0.859504,PH_q_LOGOCV_results/ESMC_results_raw/logocv_re...,PH_trained_models/host_unseen/ESMC/models_per_...,PH_trained_models/host_unseen/ESMC/models_per_...,PH_trained_models/host_unseen/ESMC/models_per_...
1,0.995,BacFormer,0.904478,0.774684,0.859504,PH_q_LOGOCV_results/BacFormer_results_raw/logo...,PH_trained_models/host_unseen/BacFormer/models...,PH_trained_models/host_unseen/BacFormer/models...,PH_trained_models/host_unseen/BacFormer/models...
2,0.990,BacFormer,0.903027,0.774667,0.851240,PH_q_LOGOCV_results/BacFormer_results_raw/logo...,PH_trained_models/host_unseen/BacFormer/models...,PH_trained_models/host_unseen/BacFormer/models...,PH_trained_models/host_unseen/BacFormer/models...
3,0.950,ESMC,0.891284,0.721186,0.834711,PH_q_LOGOCV_results/ESMC_results_raw/logocv_re...,PH_trained_models/host_unseen/ESMC/models_per_...,PH_trained_models/host_unseen/ESMC/models_per_...,PH_trained_models/host_unseen/ESMC/models_per_...
4,0.900,AF3,0.892769,0.740509,0.834711,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...
5,0.850,AF3,0.874372,0.707503,0.834711,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...
6,0.800,AF3,0.874844,0.740536,0.793388,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...
7,0.750,AF3,0.888613,0.729488,0.851240,PH_q_LOGOCV_results/AF3_results_raw/logocv_res...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...,PH_trained_models/host_unseen/AF3/models_per_q...


Task: host_unseen
Adaptation mode: full
Run: 1
Sample fraction: 1.0
Sampling seed: 46
Training seed: 1005
Device: cuda
Output: adaptation_results/138strains_46phages/routed_models/host_unseen/full
Interaction matrix: shape=(138, 46); hosts=138; phages=46


,A1a,A1b,A1c,A1d,A1e,A1f,A1g,A1h,A1i,A1j,...,S8c,S9a,S10a,S11a,S13a,S13b,S13c,S13d,S13e,S13f
Kpcas091,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Kpcas042,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Kpcas039,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CU276,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CU293,0,0,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0


Observed interactions: 6348
Adaptation interactions: 6348
Positive interactions: 153
Negative interactions: 6195


/tmp/ipykernel_341819/3201673174.py:19: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  interaction_data



Quantile: 0.999; Selected model: ESMC | Similarity groups: 121; Training groups: 120; Held-out groups: 1 per fold (121 folds)
Adaptation mode: full | trainable parameters: 1,647,617/1,647,617 (100.00%)
epoch=001; loss=1.7032; lr=9.99e-04
epoch=025; loss=0.1752; lr=5.25e-04
epoch=050; loss=0.1321; lr=5.00e-05
Adaptation mode: full | trainable parameters: 1,647,617/1,647,617 (100.00%)
epoch=001; loss=1.8262; lr=9.99e-04
epoch=025; loss=0.1677; lr=5.25e-04
epoch=050; loss=0.1077; lr=5.00e-05
Adaptation mode: full | trainable parameters: 1,647,617/1,647,617 (100.00%)
epoch=001; loss=2.0550; lr=9.99e-04
epoch=025; loss=0.1830; lr=5.25e-04
epoch=050; loss=0.1177; lr=5.00e-05
Adaptation mode: full | trainable parameters: 1,647,617/1,647,617 (100.00%)
epoch=001; loss=1.8440; lr=9.99e-04
epoch=025; loss=0.1782; lr=5.25e-04
epoch=050; loss=0.1352; lr=5.00e-05
Adaptation mode: full | trainable parameters: 1,647,617/1,647,617 (100.00%)
epoch=001; loss=2.0469; lr=9.99e-04
epoch=025; loss=0.1745; l